# NaCo - Practicum 2
**Evolution Strategies**

## Links
For those for whom the first practicum went too fast or was too difficult, we recommend taking another look at some of the core python concepts before starting this second practicum.

### Comprehensive Python tutorial:
https://www.codecademy.com/learn/learn-python-3

### Getting started with numpy
Tutorial on using numpy: https://numpy.org/doc/stable/user/quickstart.html

### Using matplotlib to create figures
For a tutorial on using matplotlib, please see: https://matplotlib.org/stable/tutorials/pyplot.html#sphx-glr-tutorials-pyplot-py

### Classes in Python
Python is an object-oriented language, and thus using classes is quite common to allow our code to be easier to re-use. For more information on classes in python, please see: https://www.w3schools.com/python/python_classes.asp

# Focus today: Evolution Strategies

In todays session, we will implement evolution strategies. For more information on the theory, please see the slides of week 3. We will be needing the following libraries:

- numpy
- matplotlib


In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt

## Exercise 1: Objective function

Just like last week, we need an objective function in order to evaluate and test our algorithm. 
This week, we will be using a two-dimensional version of the Rastrigin function.

$f(\mathbf{x}) = 10n + \sum_{i=1}^{n} [x_i^2 - 10 cos(2\pi x_i)]$

where $n$ is the dimension of the domain, which is 2 in our case. We will be using the lower bound of -1.5 and upper bound of 1.5 in this instance. 

This function has its optimum conveniently defined at $\mathbf{x}=\mathbf{0}$, but has several local optima, as we will see in our contour plots. 

For convenience, we have added a function to plot the contour of a given objective function in 2D. 
```python
def plot_landscape(
    objective_function: callable, history: list = None, delta: float = 0.025
):
    ...
```

**Your task**:

1. Define the Rastrigin function (https://en.wikipedia.org/wiki/Rastrigin_function)
2. Use it to compute the output for given input values.
3. Create a contour plot of the function.

In [2]:
def plot_landscape(
    objective_function: callable, history: list = None, delta: float = 0.025
):
    """Convenience function to plot a 2D objective function landscape

    Parameters
    ----------
    objective_function: callable
        A callable which takes a list/array of real numbers and returns a floating point value
    history: list = None
        A list of points that should be plotted in addition to the contour.
    delta: float
        The precision of the plot
    """

    x = np.arange(lb, ub + delta, delta)
    y = np.arange(lb, ub + delta, delta)
    X, Y = np.meshgrid(x, y)

    Z = np.zeros(X.shape)
    for idx1 in range(X.shape[0]):
        for idx2 in range(X.shape[1]):
            Z[idx1, idx2] = objective_function([X[idx1, idx2], Y[idx1, idx2]])

    if history is not None:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5), width_ratios=[2, 1])
    else:
        fig, ax1 = plt.subplots(figsize=(7, 5),)
        
    img = ax1.contourf(X, Y, Z, levels=100, cmap="Spectral")
    ax1.set_xlabel("x1")
    ax1.set_ylabel("x2")
    fig.colorbar(img)
    # ax1.tight_layout()

    if history is not None:
        history = np.asarray(history)
        x_best = np.argmin(history[:, 2])
        ax1.scatter(history[:, 0], history[:, 1], alpha=0.6)
        ax1.scatter(
            history[x_best, 0],
            history[x_best, 1],
            color="chartreuse",
            label=r"$x_{best}: f(x) = $" + f"{history[x_best, 2]: .3f}",
            s=50,
        )
        ax1.legend()
        
        ax2.plot(history[:, 2])
        ax2.set_ylabel(r"$f(\mathbf{x})$")
        ax2.set_xlabel("# evaluations")
        ax2.grid()        
    plt.tight_layout()

## Exersise 2: a random local search algorithm

The starting point today will be the local random search algorithm from last week. Remember, this should look like:\
**for** $i \in 1 \dots B$ \
&nbsp;&nbsp;&nbsp;&nbsp;$\mathbf{x} \sim \mathcal{U}^{n}(lb, ub)$ \
&nbsp;&nbsp;&nbsp;&nbsp;$y = f(\mathbf{x})$ 

In this example, $n$ denotes the dimension of the problem, which is 2 in this case and $\mathcal{U}^{n}(lb, ub)$ the generation of an $n$ dimensional vector of uniformly distributed random numbers, where each element is in $[lb, ub]$. 

**Note:** we do not keep track of the current best individual $\mathbf{x}^*$ in this version.


**Your Task:**
1. Implement RS
2. Plot its behaviour on the Rastrigin function

## Exersise 3: Basic mutation

We can modify our random search to not sample new points uniformly, but instead mutate the current best point slightly. As we have seen in the slides, mutation in evolution strategies permutes the current search point $\mathbf{x}^t$ by a normal distribution with zero mean and step size $\sigma$:

$\mathbf{x}^{t+1} = \mathbf{x}^t + \mathcal{N}(\mathbf{0}, \sigma)$.

For now, we'll fix the value of $\sigma$ to $||ub - lb|| / \sqrt{n}$, where $n$ is the dimension (2) and $||\cdot||$ denotes the euclidian norm, which is implemented by the numpy function ```np.linalg.norm```. 

Additionally, be sure to set an initial value of $x^t$, such that the update rule can be defined. We can do this uniformly. 

Similar to the uniform distribution, the method for using a normal (=Gaussian) distribution, can also be found in the ```numpy.random. ``` module. 

One important thing to take into account here is that we want to ensure new candidates are still within our bounds. For this, you can use the ```clip``` function from numpy (or any other method to move a point back between the bounds).

**Your Task:**
1. Set a constant value for $\sigma$
2. Set an intial value for $\mathbf{x}$
3. Modify RS to include mutation (Random Walk)
4. Plot its behaviour on the Rastrigin function

## Exercise 4: Add a population of individuals
Since the evolution strategy is a population based algorithm, in this exercise we'll add a population of search points to the algorithm. The number of individuals in the population is denoted by the greek letter $\lambda$ for which the value 10 should be used here. 

We be implementing a simplified $(1, \lambda)$-ES in this exercise, which means that one of the $\lambda$ individuals should be kept as starting point for our next generation. We'll name this point $\mathbf{x}'$, which we initialize uniformly at random, similar to x in the previous exersice. In our optimization loop, we set the $\mathbf{x}'$ to the mutated $\mathbf{x}_i$ which has the best (minimal) objective function value.

In order to do this, we need to keep track of the index of the current best solution within a generation. 

**Your Task:**
1. Rename $x$ to $x'$
2. Create a population of 10 individuals 
   1. Each point should be mutated from $x'$, and evaluated on $f$
3. Keep track of the best in the population (lowest $f(x)$)
4. Set $x'$ to the best individual in the population
5. Run and plot the behaviour

## Exercise 5: Self-adaptation

As discussed in the lecture, the defining property of evolution strategies is their ability for self-adaptation, that is, their ability to self-adapt their strategy parameters during optimization. 

As you probably noticed in the contour plot in the previous exercises, a great many of the sampled points were sampled out of bounds (and thus placed on the edge of the search domain by the clip function).

This is due to the fact that we used a static value for the strategy parameter $\sigma$, which remains large throughout the search process, and thus the algorithm is never able to change its search behaviour to be more localized. 

In this exercise we'll add self-adaptation for $\sigma$, to the code you previously implemented. The value of the step size $\sigma$ should be endogenous, which means that for each individual $\mathbf{x}_i$ we should have a *specific* $\sigma_i$ which is also mutated. After mutation, we set value of $\sigma'$ to the best mutated value of all $\sigma_i$, similar to how we set $\mathbf{x}'$ in the previous exercise. 

As shown in the slides, this changes the mutation to the following set of equations:

$\sigma_i = \sigma' \cdot \exp(\tau_0 \cdot \mathcal{N}(0,1))$

$\mathbf{x}_i = \mathbf{x}' + \mathcal{N}(\mathbf{0}, \sigma_i)$

Where the learning rate parameter $\tau_0 = \frac{1}{\sqrt{n}}$ according to Schwefel recommendation (see slides).

**Your Task:**
1. Define $\tau_0$
2. Rename $\sigma$ to $\sigma'$
3. Add $\sigma_i$ to each individual $i$
4. Mutate $\sigma_i$, and mutate $\mathbf{x}_i$ with $\sigma_i$
6. Set $\sigma'$ similar as how you set $\mathbf{x}'$ 
7. Run and plot the behaviour

## Exercise 6: Selection and Recombination
For the case where $\mu > 1$, i.e. when we consider $\mu$ parents instead of only parent $\mathbf{x}'$ as was done up until now, we need a new way of generating offspring. We require a *recombination* operator, that takes the genetic information of two or more parents, and recomines that information into a single new offspring. 

In this exercise, we set the value of $\mu$ to 4, and implement local intermediate recombination between two parent solutions. This boils down to taking the arthimetic average between the values of two randomly selected parents. Note that in addition to taking the average $\mathbf{x}_i$ values, we should also take the arithmetic average of each $\sigma_i$. 

In order to determine which of the $\lambda$ offspring should be used as parents, we need to define a selection procedure. In evolution strategies, selection is deterministic, which is fancy way of saying that we simply take the $\mu$ best solutions of our generated offspring. Therefore, at the end of each generation, sort the population based on their function values (ascending), and take the first $\mu$ solutions as the next parent population. 

**Note:** we are implementing $(\mu, \lambda)$-selection here.

**Your Task**
1. Define a parent population of size $\mu$
2. Implement recombination (local intermediate):
   1. Randomly pick two parents and set the offspring to the average (also for $\sigma$!)
3. Implement deterministic selection
   1. Sort the offspring population, and pick the $\mu$ best
4. Plot the behaviour 

## Exercise 7: clean up the code and turn it into a class

Python is an object-oriented language, and thus using classes is quite common to allow our code to be easier to re-use. For more infomration on classes in python, please see: https://www.w3schools.com/python/python_classes.asp

We can now turn our ES-code into a class. 
We will make use of the default functions: ```__init__``` for initialization, which we can use to e.g. set a starting step size, and ```__call__``` for doing the optimization. This call function will get passed the objective function, and returns the history list. Try add member functions for the three operations: recombination, mutation and selection. 

It is up to you to decide where you pass in the variables relating to problem dimensionality and bounds.

**Note:** It would be good to add separate functions for each of the operators; e.g. mutation, recombination, selection.

### Repeat & Reuse

With the class defined, we can instantiate it to make an object which we can use to do the optimization. With this in place, we can now try different objective functions, or different parameter settings. For now, run the algorithm once through a class based interface to reproduce the output of the previous exercise.